# Project Notebook 2: Computer Vision Model

**Author:** Vansh Goel
**Date:** October 21, 2025

## 1. Introduction

This notebook fulfills the Computer Vision requirement of the assignment. Given the small dataset size (312 images), training a deep CNN/ResNet from scratch would lead to severe overfitting.

Therefore, this notebook implements a modern and highly effective **Zero-Shot Classification** model using OpenAI's **CLIP** (loaded via the `transformers` library).

## 2. Zero-Shot Classification

Instead of training, this model uses its pre-trained knowledge of images and text. We provide it an image and a list of candidate labels (e.g., "Furniture", "Hardware"). The model compares the image to the text labels and returns a similarity score, effectively classifying the image without ever being explicitly trained on our specific categories. As shown in the tests below, this approach is fast and surprisingly accurate.

---

In [1]:
import pandas as pd
import os
import ast # A library to safely evaluate string-lists

print(f"Current working directory: {os.getcwd()}")
file_path = '../data/products.csv' 

try:
    df = pd.read_csv(file_path)
    print(f"✅ Data loaded successfully! Total rows: {df.shape[0]}")
    
    # --- Clean up the 'images' column ---
    # The 'images' column is a string that looks like a list.
    # We need to safely parse it.
    
    # 1. Fill missing image data with an empty list string
    df['images'].fillna('[]', inplace=True)
    
    # 2. Use 'ast.literal_eval' to convert the string '[]' into a real list []
    df['image_list'] = df['images'].apply(ast.literal_eval)
    
    # 3. Create a new column 'first_image_url' with just the *first* image
    # We'll use this first image to train our model
    df['first_image_url'] = df['image_list'].apply(lambda x: x[0].strip() if x else None)
    
    print("✅ 'images' column parsed.")

    # --- Let's look at our key columns ---
    print("\n--- CV Task Data Check ---")
    print(df[['categories', 'first_image_url']].head())
    
    # --- How many images do we actually have? ---
    print(f"\nTotal products with a valid first image: {df['first_image_url'].notna().sum()}")
    print(f"Total products missing an image: {df['first_image_url'].isna().sum()}")

except Exception as e:
    print(f"❌ An error occurred: {e}")

Current working directory: c:\Users\Asus\Desktop\product-recommendation-app\notebooks
✅ Data loaded successfully! Total rows: 312
✅ 'images' column parsed.

--- CV Task Data Check ---
                                          categories  \
0  ['Home & Kitchen', 'Storage & Organization', '...   
1  ['Home & Kitchen', 'Furniture', 'Dining Room F...   
2  ['Patio, Lawn & Garden', 'Outdoor Décor', 'Doo...   
3  ['Patio, Lawn & Garden', 'Outdoor Décor', 'Doo...   
4  ['Home & Kitchen', 'Furniture', 'Game & Recrea...   

                                     first_image_url  
0  https://m.media-amazon.com/images/I/416WaLx10j...  
1  https://m.media-amazon.com/images/I/31SejUEWY7...  
2  https://m.media-amazon.com/images/I/41RgefVq70...  
3  https://m.media-amazon.com/images/I/61vz1Igler...  
4  https://m.media-amazon.com/images/I/41p4d4VJnN...  

Total products with a valid first image: 312
Total products missing an image: 0


C:\Users\Asus\AppData\Local\Temp\ipykernel_23328\51545742.py:17: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['images'].fillna('[]', inplace=True)


In [2]:
import requests
from PIL import Image
from transformers import pipeline
import ast

print("✅ Libraries imported.")

# --- 1. Load the Zero-Shot Image Classification Model ---
# This will download the model (CLIP) the first time you run it.
print("Loading CLIP zero-shot classification pipeline...")
classifier = pipeline("zero-shot-image-classification", model="openai/clip-vit-base-patch32")
print("✅ Model loaded.")


# --- 2. Prepare Sample Data ---

# We need a list of candidate labels. Let's create a simple one
# by looking at the 'categories' column.
# First, let's parse the categories column properly
df['categories_list'] = df['categories'].apply(ast.literal_eval)

# Now, let's get the *main* category (the second item in the list, e.g., 'Furniture')
df['main_category'] = df['categories_list'].apply(lambda x: x[1] if len(x) > 1 else x[0])

# Get a list of the top 10 most common main categories
candidate_labels = df['main_category'].value_counts().nlargest(10).index.tolist()

print(f"\nTop 10 categories to use as labels: {candidate_labels}")


# --- 3. Test the Model on a Few Samples ---
print("\n--- Running Classification Test ---")

# Get 3 sample products to test
samples = df.sample(3)

for index, row in samples.iterrows():
    image_url = row['first_image_url']
    actual_category = row['main_category']
    
    print(f"\nTest Item (ID: {row['uniq_id']})")
    print(f"  Image URL: {image_url}")
    print(f"  ACTUAL Category: {actual_category}")
    
    try:
        # Open the image from the URL
        image = Image.open(requests.get(image_url, stream=True).raw)
        
        # Run the classification
        predictions = classifier(image, candidate_labels=candidate_labels)
        
        # Print the top prediction
        top_prediction = predictions[0]
        print(f"  PREDICTED Category: {top_prediction['label']} (Score: {top_prediction['score']:.2f})")
        
    except Exception as e:
        print(f"  Could not process image. Error: {e}")

c:\Users\Asus\Desktop\product-recommendation-app\backend\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Libraries imported.
Loading CLIP zero-shot classification pipeline...


c:\Users\Asus\Desktop\product-recommendation-app\backend\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Asus\.cache\huggingface\hub\models--openai--clip-vit-base-patch32. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installe

✅ Model loaded.

Top 10 categories to use as labels: ['Furniture', 'Outdoor Décor', 'Storage & Organization', 'Home Décor Products', 'Hardware', 'Nursery', 'Television & Video', 'Tools & Accessories', 'Bath', 'Kitchen & Dining']

--- Running Classification Test ---

Test Item (ID: 6259d40c-b254-5fac-aacc-f0c7b0caad7f)
  Image URL: https://m.media-amazon.com/images/I/31dCSKQ14YL._SS522_.jpg
  ACTUAL Category: Furniture
  PREDICTED Category: Furniture (Score: 0.91)

Test Item (ID: eecadc04-afeb-5858-a60f-3979a80b79ba)
  Image URL: https://m.media-amazon.com/images/I/41aRwocdfAL._SS522_.jpg
  ACTUAL Category: Furniture
  PREDICTED Category: Home Décor Products (Score: 0.75)

Test Item (ID: de8420e9-0b53-5471-9e25-32cdfb012603)
  Image URL: https://m.media-amazon.com/images/I/41ZBS1hvHzL._SS522_.jpg
  ACTUAL Category: Furniture
  PREDICTED Category: Furniture (Score: 0.85)
